In [2]:
import glob
import math
import numpy as np
import ROOT
from utils.hist_utils import get_highest_channels, fit_and_scale_mc

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/Include/Codes/")

ROOT.gInterpreter.Declare('#include "const.h"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

In [3]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/*.root"
)

all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS2_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS3_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/hal/DATA_THREE_PI0_NoSmearing/*.root")

def get_valid_files(file_list):
    valid = []
    for f in file_list:
        try:
            tfile = ROOT.TFile.Open(f, "READ")
            if tfile and not tfile.IsZombie() and tfile.GetNkeys() > 0:
                valid.append(f)
            else:
                print(f"Pomijam uszkodzony/pusty plik: {f}")
            if tfile:
                tfile.Close()
        except Exception:
            print(f"Błąd otwarcia pliku (pomijam): {f}")
    return valid

all_files = get_valid_files(all_files)

# Scaling factor
lumi_data = 239811.5318
lumi_mc = 351638.6865
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_3pi0 = df.Filter("mcflag == 1 && (mctruth == 4 || mctruth == 0 || mctruth == -1)")
df_data = (df
    .Define("Kchrec_5", "Kchrec[5]")
    .Define("Knerec_5", "Knerec[5]")
    .Define("KnerecSix_5", "KnerecSix[5]")
    .Define("MassPi01", "pi01Fit[5]")
    .Define("MassPi02", "pi02Fit[5]")
    .Define("rho_ch", "hypot(Kchrec[6] - ip[0], Kchrec[7] - ip[1])")
    .Define("rho_ne", "hypot(Knerec[6] - ip[0], Knerec[7] - ip[1])")
    .Define("z_ch", "Kchrec[8] - ip[2]")
    .Define("rho_fid_omega", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_fid_omega", "Kchrec[8] - Knerec[8]")
    .Filter("mcflag == 0 && mctruth != -1"))
df = (df
    .Define("Kchrec_5", "Kchrec[5]")
    .Define("Knerec_5", "Knerec[5]")
    .Define("KnerecSix_5", "KnerecSix[5]")
    .Define("MassPi01", "pi01Fit[5]")
    .Define("MassPi02", "pi02Fit[5]")
    .Define("rho_ch", "hypot(Kchrec[6] - ip[0], Kchrec[7] - ip[1])")
    .Define("rho_ne", "hypot(Knerec[6] - ip[0], Knerec[7] - ip[1])")
    .Define("z_ch", "Kchrec[8] - ip[2]")
    .Define("rho_fid_omega", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_fid_omega", "Kchrec[8] - Knerec[8]")
    .Filter("mctruth != -1 && mctruth != 0 "))

Błąd otwarcia pliku (pomijam): /workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/mk0_initial_analysis_all_phys_THREE_PI0_NoSmearing_3pi0_NONE_205.root
Lumi factor: 0.6820


Warning in <TFile::Init>: file /workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/mk0_initial_analysis_all_phys_THREE_PI0_NoSmearing_3pi0_NONE_205.root probably not closed, trying to recover
Warning in <TFile::Init>: no keys recovered, file has been made a Zombie


In [4]:
counts = []

counts.append(df_3pi0.Count())
counts.append(df_3pi0.Filter("mctruth == 4 || mctruth == 0").Count())
counts.append(df_3pi0.Filter("mctruth == 4").Count())

errors = [300, 301, 303, 307, 308]
counts_errors = []
combined_error = ""

for error in errors:
    combined_error += f" && errorcode != {error}" if len(combined_error) > 0 else f"errorcode != {error}"
    counts_errors.append(df_3pi0.Filter(combined_error).Count())

all_counts = counts + counts_errors

ROOT.RDF.RunGraphs(all_counts)

for i, val in enumerate(counts):
    counts[i] = val.GetValue()

error_eff = []
rel_error_eff = []

for i, val in enumerate(counts_errors):
    counts_errors[i] = val.GetValue()
    eff = counts_errors[i] / counts[0] if counts[0] > 0 else 1.0
    rel_eff = counts_errors[i] / counts_errors[i-1] if i > 0 else eff
    error_eff.append(eff)
    rel_error_eff.append(rel_eff)

efficiencies = [0, 0, 0]
efficiencies[0] = counts[1] / counts[0] if counts[0] > 0 else 1.0
efficiencies[1] = counts[2] / counts[1] if counts[1] > 0 else 1.0
efficiencies[2] = counts[2] / counts[0] if counts[0] > 0 else 1.0

print(f"Counts total: {counts[0]},\nCounts preselection: {counts[1]},\nCounts selection: {counts[2]}")
print(f"Efficiency total: {efficiencies[2]*100:.2f}%,\nEfficiency preselection: {efficiencies[0]*100:.2f}%,\nEfficiency selection: {efficiencies[1]*100:.2f}%")

for i, eff in enumerate(error_eff):
    print(f"Error {errors[i]}: {eff*100:.2f}% ({rel_error_eff[i]*100:.2f}%) ({counts_errors[i]})")

Counts total: 25292412,
Counts preselection: 18321552,
Counts selection: 18321552
Efficiency total: 72.44%,
Efficiency preselection: 72.44%,
Efficiency selection: 100.00%
Error 300: 90.53% (90.53%) (22897171)
Error 301: 75.22% (83.09%) (19025415)
Error 303: 74.31% (98.79%) (18794286)
Error 307: 72.44% (97.48%) (18321552)
Error 308: 72.44% (100.00%) (18321552)


In [5]:
histos = {}

for mctruth, chann in channName.items():
    if mctruth > 0 and mctruth < 8:
        hist_list = []
        hist_list.append((df.Filter(f"mctruth == {mctruth}")
                           .Histo1D(("h_rho_ch", ";#rho_{ch} [cm];Counts", 201, 0, 12), "rho_ch")))
        hist_list.append((df.Filter(f"mctruth == {mctruth}")
                           .Histo1D(("h_z_ch", ";z_{ch} - z_{IP} [cm];Counts", 201, -6, 6), "z_ch")))

        histos[chann] = hist_list

handles = [h for hist_list in histos.values() for h in hist_list]
        
ROOT.RDF.RunGraphs(handles)

1

In [6]:
highest_chann = get_highest_channels(histos)

In [7]:
ROOT.gStyle.SetOptStat(0)

canvadivision = ROOT.TCanvas("canva", "", 800, 400)
canvadivision.Divide(2, 1)

num_pads = len(next(iter(histos.values())))
legends = []  # Zapobiega usunięciu legend przez Python GC

for i in range(num_pads):
    canvadivision.cd(i + 1)
    
    # Tworzymy legendę na padzie (x1, y1, x2, y2)
    leg = ROOT.TLegend(0.65, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)  # Przezroczyste tło
    leg.SetTextSize(0.035)
    
    # Sortujemy tak, aby najwyższy kanał dla danego pada był na samym początku
    sorted_channels = sorted(
        histos.items(),
        key=lambda item: item[0] == highest_chann[i],
        reverse=True
    )

    for idx, (chann, hist_list) in enumerate(sorted_channels):
        hist = hist_list[i]
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        
        draw_option = "HIST" if idx == 0 else "SAME HIST"

        color = channColor.get(chann, ROOT.kBlack)
        h.SetLineColor(color)
        h.SetLineWidth(2)
        h.Draw(draw_option)
        
        # Wypełnianie legendy (pobieramy ładny tytuł z channTitle, a jeśli brakuje — używamy chann)
        label = channTitle.get(chann, chann)
        leg.AddEntry(h, str(label), "l")

    # Rysujemy legendę na padzie i zapisujemy referencję
    leg.Draw()
    legends.append(leg)

canvadivision.Draw()

## Optimization of the cut for the control sample tag $K\to\pi^+\pi^-$

In [8]:
df_CS_tag = {}

df_CS_tag["000"] = df.Filter("has000 && mctruth != 0 && mctruth != -1")
df_CS_tag["non000"] = df.Filter("!has000 && mctruth != 0 && mctruth != -1")
df_CS_tag["data"] = df_data.Filter("mctruth != -1")

total_counts = {}

total_counts["000"] = df_CS_tag["000"].Count().GetValue()
total_counts["non000"] = df_CS_tag["non000"].Count().GetValue()
total_counts["data"] = df_CS_tag["data"].Count().GetValue()

### Create the histograms

In [9]:
cut_cond = "true"
optimized_variable = "bestErrorSixGamma"
var_title = "#Delta_{6} [cm^{2}]"

In [10]:
basic_hist = {}
modes = {"full": [501, 0, 5000], "zoom": [501, 0, 3000]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["000", "non000", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: PM, Index 1: nonPM, Index 2: data
    
    # Skalowanie histogramów MC (PM i nonPM)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (PM + nonPM) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [11]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 2000, 1000)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow3#pi^{0}", 
    "Events without primary K#rightarrow3#pi^{0}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 1.05
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


### Optimization loop

In [12]:
cut_values = [rms * float(val) for val in np.arange(0, 5.0, 0.1)]

pm_counts = []
nonpm_counts = []

for val in cut_values:
    condition = f"{optimized_variable}< {val}"

    pm_counts.append(df_CS_tag["000"].Filter(condition).Count())
    nonpm_counts.append(df_CS_tag["non000"].Filter(condition).Count())

In [13]:
all_counts = pm_counts + nonpm_counts

ROOT.RDF.RunGraphs(all_counts)

1

In [14]:
pm_counts = [val.GetValue() for val in pm_counts]
nonpm_counts = [val.GetValue() for val in nonpm_counts]

In [15]:
purity = []
significance = []
efficiency = {"000": [], "non000": []}

for nonpm, pm in zip(nonpm_counts, pm_counts):
    purity.append(pm / (nonpm + pm) if (nonpm + pm) > 0 else 0.99)
    significance.append(pm / np.sqrt(nonpm + pm) if (nonpm + pm) > 0 else 0.0)
    efficiency["000"].append(pm / total_counts["000"]) 
    efficiency["non000"].append(nonpm / total_counts["non000"]) 

In [16]:
canva = ROOT.TCanvas("canva", "", 800, 800)

x_arr = np.array(cut_values, dtype=float)
y_arr = np.array(purity, dtype=float)

effpm_arr = np.array(efficiency["000"], dtype=float)
effnonpm_arr = np.array(efficiency["non000"], dtype=float)

graph = ROOT.TGraph(len(x_arr), x_arr, y_arr)
graph_pm = ROOT.TGraph(len(x_arr), x_arr, effpm_arr)
graph_nonpm = ROOT.TGraph(len(x_arr), x_arr, effnonpm_arr)
graph.SetMaximum(1.2)
graph.SetMinimum(0.0)

graph.SetTitle("")
graph.GetXaxis().SetTitle("Cut value #Delta_{6} [cm^{2}]")
graph.GetYaxis().SetTitle("Value [-]")
graph.Draw("AL")

graph_pm.SetLineColor(ROOT.kRed)
graph_pm.Draw("LSAME")

graph_nonpm.SetLineColor(ROOT.kBlue)
graph_nonpm.Draw("LSAME")

line = ROOT.TLine(200.0, 0.0, 200.0, 1.2)
line.SetLineStyle(2)
line.Draw("SAME")

legend = ROOT.TLegend(0.5, 0.2, 0.85, 0.45)
legend.AddEntry(graph, "Purity K#rightarrow3#pi^{0}", "l")
legend.AddEntry(graph_pm, "Efficiency K#rightarrow3#pi^{0}", "l")
legend.AddEntry(graph_nonpm, "Efficiency non K#rightarrow3#pi^{0}", "l")
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw()
canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


### Checks of counts, purity and efficiency in the optimal threshold

In [17]:
optimal_purity = graph.Eval(200.0)
optimal_efficiency = graph_pm.Eval(200.0)
nonpm_eff = graph_nonpm.Eval(200.0)

print(f"Optimal purity: {optimal_purity*100:.2f}%")
print(f"Optimal efficiency: {optimal_efficiency*100:.2f}%")
print(f"NonPM efficiency: {nonpm_eff*100:.2f}%")

Optimal purity: 98.35%
Optimal efficiency: 4.51%
NonPM efficiency: 0.05%


In [18]:
pm_counts_arr = np.array(pm_counts, dtype=float)
nonpm_counts_arr = np.array(nonpm_counts, dtype=float)

graph_pm_counts = ROOT.TGraph(len(x_arr), x_arr, pm_counts_arr)
graph_nonpm_counts = ROOT.TGraph(len(x_arr), x_arr, nonpm_counts_arr)

# ---

optimal_counts_pm = graph_pm_counts.Eval(200.0)
optimal_counts_nonpm = graph_nonpm_counts.Eval(200.0)

print(f"Optimal counts pm: {optimal_counts_pm:.0f}")
print(f"Optimal counts non pm: {optimal_counts_nonpm:.0f}")

Optimal counts pm: 880017
Optimal counts non pm: 14828


In [19]:
tag_condition = "bestErrorSixGamma < 200.0"

df_CS_tag["000"] = df_CS_tag["000"].Filter(tag_condition)
df_CS_tag["non000"] = df_CS_tag["non000"].Filter(tag_condition)
df_CS_tag["data"] = df_CS_tag["data"].Filter(tag_condition)

total_counts = {}

total_counts["000"] = df_CS_tag["000"].Count().GetValue()
total_counts["non000"] = df_CS_tag["non000"].Count().GetValue()
total_counts["data"] = df_CS_tag["data"].Count().GetValue()

### Create the histograms

In [20]:
cut_cond = "true"
optimized_variable = "bestErrorSixGamma"
var_title = "#Delta_{6} [cm^{2}]"

In [21]:
basic_hist = {}
modes = {"full": [501, 0, 5000], "zoom": [501, 0, 3000]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["000", "non000", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: PM, Index 1: nonPM, Index 2: data
    
    # Skalowanie histogramów MC (PM i nonPM)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (PM + nonPM) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [22]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 1000, 500)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow3#pi^{0}", 
    "Events without primary K#rightarrow3#pi^{0}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 1.05
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


## Conditional probability: KL->3pi0 if KS->pi+pi- (change of lookup to channel level)

In [23]:
df_tag_channel = {}

for mctruth, chann in channName.items():
    if chann == 'Data':
        df_tag_channel[chann] = df_data.Filter(tag_condition)
    elif chann != 'MC sum':
        df_tag_channel[chann] = df.Filter(f"mctruth == {mctruth} && ({tag_condition})")

In [24]:
variables = ["Chi2SignalKinFit", "Knerec_5", "KnerecSix_5", "MassPi01", "MassPi02", "Kchrec_5"]
var_title = ["#chi^{2}_{fit, +-00} [-]", 
             "m^{inv}_{4#gamma} [MeV/c^{2}]", 
             "m^{inv}_{6#gamma} [MeV/c^{2}]", 
             "m^{inv}_{2#gamma,1} [MeV/c^{2}]",
             "m^{inv}_{2#gamma,2} [MeV/c^{2}]",
             "m^{inv}_{+-} [MeV/c^{2}]"]

In [25]:
basic_hist = {}
modes = {
         "full": [[501, 0, 1000], [501, 0, 1000], [501, 2, 1000], [501, 0, 500], [501, 0, 500], [501, 280, 600]], 
         "zoom": [[50, 0, 100], [251, 100, 600], [171, 300, 600], [251, 0, 300], [251, 0, 300], [501, 490, 505]]
        }

for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}
    
    for mode, pars in modes.items():
        
        hist_dict = {}
        for chann in channName.values():
            if chann == "MC sum":
                continue
            h_title = f"hist_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"
            hist_dict[chann] = df_tag_channel[chann].Histo1D((h_title, hist_title_list, *pars[idx]), var)

        mode_dict[mode] = hist_dict

    basic_hist[var] = mode_dict
            
# 2. Ewaluacja wszystkich grafów na raz (Lazy Evaluation)
all_ptrs = [
    h_ptr 
    for mode_d in basic_hist.values() 
    for ch_d in mode_d.values() 
    for h_ptr in ch_d.values()
]
ROOT.RDF.RunGraphs(all_ptrs)

# 3. Skalowanie MC i tworzenie "MC sum"
for var, mode_d in basic_hist.items():
    for mode, ch_d in mode_d.items():
        
        # Skalowanie kanałów MC (wszystko poza 'data')
        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        # Tworzenie sumy MC (np. łącząc kanały PM i nonPM)
        # Przyjmujemy, że pierwszy kanał MC tworzy bazę
        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]
        
        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)
            
            # Zapisujemy wygenerowaną sumę w słowniku
            ch_d["MC sum"] = h_mc_sum

In [26]:
# Określamy wymiary siatki
n_rows = len(variables)
n_cols = len(modes)

# Tworzymy płótno przeskalowane do liczby wierszy i kolumn
canva = ROOT.TCanvas("canva", "Histograms", 500 * n_cols, 400 * n_rows)
canva.Divide(n_cols, n_rows)

stacks = []
legends = []

# Pętla po wierszach (zmienne) i kolumnach (tryby)
for row_idx, (var, x_title) in enumerate(zip(variables, var_title)):
    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):
        
        # Obliczenie indeksu podokna w TCanvas (ROOT numeruje od 1: wiersz po wierszu)
        pad_idx = row_idx * n_cols + col_idx + 1
        canva.cd(pad_idx)
        
        hs = ROOT.THStack(f"hs_{var}_{mode_name}", f"{mode_name.capitalize()} mode;{x_title};Counts")
        leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        
        data_hist = None

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            color = channColor.get(chann_key, ROOT.kBlack)
            title = channTitle.get(chann_key, chann_key)
            
            if chann_key == "Data":
                # Styl dla danych eksperymentalnych (PE1)
                h.SetMarkerStyle(20)
                h.SetMarkerSize(0.8)
                h.SetMarkerColor(color)
                h.SetLineColor(color)
                h.SetLineWidth(2)
                
                data_hist = h
                leg.AddEntry(h, title, "pe")
            else:
                # Styl dla MC
                h.SetLineColor(color)
                h.SetLineWidth(2)
                
                hs.Add(h)
                leg.AddEntry(h, title, "l")

        # 1. Rysujemy stack z MC
        hs.Draw("nostack hist")
        
        # 2. Rysujemy Dane oraz dopasowujemy skalę Y
        if data_hist:
            data_hist.Draw("E1 P SAME")
            max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
            hs.SetMaximum(max_y_val)

        leg.Draw()

        stacks.append(hs)
        legends.append(leg)

# Zapobieganie usunięciu obiektów z pamięci przez Python Garbage Collector
canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


## Application of cuts

In [27]:
import math

pi0Mass1_mean = 134.83240924168848
pi0Mass2_mean = 134.87134080668446

for chann, df_part in df_tag_channel.items():
    df_tag_channel[chann] = (df_part
                .Define("u", f"((MassPi01 - {pi0Mass1_mean}) + (MassPi02 - {pi0Mass2_mean})) / sqrt(2)")
                .Define("v", f"((MassPi01 - {pi0Mass1_mean}) - (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"))

In [28]:
ROOT.gInterpreter.Declare("""
bool ellipse_cut(double u, double v) {
    double u0 = -92.9524, v0 = 7.1644, su = 33.7633, sv = 42.0163, rho = 0.556;
    double n_sigma_cut = 2.5;
    
    double du = u - u0;
    double dv = v - v0;
    double inv_rho2 = 1.0 / (1.0 - rho * rho);
    double dist2 = inv_rho2 * ((du * du) / (su * su) + (dv * dv) / (sv * sv) - 2.0 * rho * (du * dv) / (su * sv));
    return std::sqrt(std::max(0.0, dist2)) > n_sigma_cut;
}
""")

True

In [29]:
# # 2. Add your cuts, including any that use u and v
# cuts = [
#     "Chi2SignalKinFit < 30.0",
#     "ellipse_cut(Knerec_5 - 497.611, KnerecSix_5 - 497.611)",
#     "abs(u) < 8.085754304 && abs(v) < 11.514665004",
#     "(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45 && sqrt(rho_ch*rho_ch + rho_ne*rho_ne) > 0.8) || !(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45)"
# ]

# cut_counts = []
# cut_counts_bkg = []
# cut_eff = []
# rel_cut_eff = []

# purity = []

# combined_cut = ""

# for cut in cuts:
#     combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
#     cut_counts.append(df_tag_3pi0.Filter(combined_cut).Count())
#     cut_counts_bkg.append(df_tag_non_3pi0.Filter(combined_cut).Count())

# tot_cuts = cut_counts + cut_counts_bkg
# ROOT.RDF.RunGraphs(tot_cuts)

# for i, count in enumerate(cut_counts):
#     cut_counts[i] = count.GetValue()
#     cut_counts_bkg[i] = cut_counts_bkg[i].GetValue()
#     eff = cut_counts[i] / basis_3pi0_counts if basis_3pi0_counts > 0 else 1.0
#     rel_eff = cut_counts[i] / cut_counts[i-1] if i > 0 else eff
#     pur = (cut_counts[i]) / (cut_counts[i] + cut_counts_bkg[i])
#     cut_eff.append(eff)
#     rel_cut_eff.append(rel_eff)
#     purity.append(pur)

#     print(f"Cut {cuts[i]}: Counts {int(cut_counts[i]*scaling_factor)}, Efficiency {cut_eff[i]*100:.3f}%, Relative efficiency {rel_cut_eff[i]*100:.3f}%, Counts {int(cut_counts_bkg[i]*scaling_factor)}, Purity {purity[i]*100:.3f}%")



In [30]:
# # 2. Add your cuts, including any that use u and v

# cut_counts = []
# cut_eff = []
# rel_cut_eff = []

# combined_cut = ""

# for cut in cuts:
#     combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
#     cut_counts.append(df_tag_data.Filter(combined_cut).Count())

# ROOT.RDF.RunGraphs(cut_counts)

# for i, count in enumerate(cut_counts):
#     cut_counts[i] = count.GetValue()
#     eff = cut_counts[i] / basis_data_counts if basis_data_counts > 0 else 1.0
#     rel_eff = cut_counts[i] / cut_counts[i-1] if i > 0 and cut_counts[i-1] > 0 else eff
#     cut_eff.append(eff)
#     rel_cut_eff.append(rel_eff)

#     print(f"Cut {cuts[i]}: Counts {cut_counts[i]}, Efficiency {cut_eff[i]*100:.3f}%, Relative efficiency {rel_cut_eff[i]*100:.3f}%")

In [31]:
variables = ["Chi2SignalKinFit", "Knerec_5", "KnerecSix_5", "MassPi01", "MassPi02", "Kchrec_5"]
var_title = ["#chi^{2}_{fit, +-00} [-]", 
             "m^{inv}_{4#gamma} [MeV/c^{2}]", 
             "m^{inv}_{6#gamma} [MeV/c^{2}]", 
             "m^{inv}_{2#gamma,1} [MeV/c^{2}]",
             "m^{inv}_{2#gamma,2} [MeV/c^{2}]",
             "m^{inv}_{+-} [MeV/c^{2}]"]

In [32]:
combined_cut = '''abs(Kchrec_5 - 497.605) < 2.637 && Chi2SignalKinFit < 30.0'''

#'''Chi2SignalKinFit < 30. && 
#                  ellipse_cut(Knerec_5 - 497.611, KnerecSix_5 - 497.611) && 
#                  (abs(u) < 8.085754304 && abs(v) < 11.514665004) && 
#                  ((rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45 && sqrt(rho_ch*rho_ch + rho_ne*rho_ne) > 0.8) || !(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45))
#                '''

basic_hist = {}
modes = {
         "full": [[501, 0, 1000], [501, 0, 1000], [501, 2, 1000], [501, 0, 500], [501, 0, 500], [501, 280, 600]], 
         "zoom": [[101, 0, 100], [251, 100, 600], [171, 300, 600], [251, 0, 300], [251, 0, 300], [501, 490, 505]]
        }

for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}
    
    for mode, pars in modes.items():
        
        hist_dict = {}
        for chann in channName.values():
            if chann == "MC sum":
                continue
            h_title = f"hist_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"
            hist_dict[chann] = df_tag_channel[chann].Filter(combined_cut).Histo1D((h_title, hist_title_list, *pars[idx]), var)

        mode_dict[mode] = hist_dict

    basic_hist[var] = mode_dict
            
# 2. Ewaluacja wszystkich grafów na raz (Lazy Evaluation)
all_ptrs = [
    h_ptr 
    for mode_d in basic_hist.values() 
    for ch_d in mode_d.values() 
    for h_ptr in ch_d.values()
]
ROOT.RDF.RunGraphs(all_ptrs)

# 3. Skalowanie MC i tworzenie "MC sum"
for var, mode_d in basic_hist.items():
    for mode, ch_d in mode_d.items():
        
        # Skalowanie kanałów MC (wszystko poza 'data')
        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        # Tworzenie sumy MC (np. łącząc kanały PM i nonPM)
        # Przyjmujemy, że pierwszy kanał MC tworzy bazę
        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]
        
        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)
            
            # Zapisujemy wygenerowaną sumę w słowniku
            ch_d["MC sum"] = h_mc_sum

In [33]:
canvases = []
stacks = []
legends = []

# Pętla po zmiennych – każda zmienna tworzy nowy Canvas
for row_idx, (var, x_title) in enumerate(zip(variables, var_title)):

    # Tworzenie osobnego TCanvas podzielonego na kolumny dla poszczególnych trybów (mode)
    canva = ROOT.TCanvas(f"canva_{var}", f"Canvas {var}", 500 * n_cols, 400)
    canva.Divide(n_cols, 1)

    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):

        canva.cd(col_idx + 1)

        hs = ROOT.THStack(
            f"hs_{var}_{mode_name}",
            f"{mode_name.capitalize()} mode;{x_title};Counts",
        )
        leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)

        data_hist = None

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            color = channColor.get(chann_key, ROOT.kBlack)
            title = channTitle.get(chann_key, chann_key)

            if chann_key == "Data":
                # Styl dla danych eksperymentalnych (PE1)
                h.SetMarkerStyle(20)
                h.SetMarkerSize(0.8)
                h.SetMarkerColor(color)
                h.SetLineColor(color)
                h.SetLineWidth(2)

                data_hist = h
                leg.AddEntry(h, title, "pe")
            else:
                # Styl dla MC
                h.SetLineColor(color)
                h.SetLineWidth(2)

                hs.Add(h)
                leg.AddEntry(h, title, "l")

        # 1. Rysujemy stack z MC
        hs.Draw("nostack hist")

        # 2. Rysujemy Dane oraz dopasowujemy skalę Y
        if data_hist:
            data_hist.Draw("E1 P SAME")
            max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
            hs.SetMaximum(max_y_val)

        leg.Draw()

        stacks.append(hs)
        legends.append(leg)

    canva.Draw()
    canvases.append(canva)

# Zapobieganie usunięciu obiektów z pamięci przez Python Garbage Collector
for c in canvases:
    c._stacks = stacks
    c._legends = legends